In [4]:
import pandas as pd
import arff
import os

# Build the absolute path
project_root = '/workspaces/PhishTrace'
filepath = os.path.join(project_root, 'data/raw/Training Dataset.arff')

print(f"Looking for: {filepath}")
print(f"File exists: {os.path.exists(filepath)}")

# Load the ARFF file
with open(filepath, 'r') as f:
    dataset = arff.load(f)

# Convert to pandas DataFrame
df = pd.DataFrame(dataset['data'], columns=[attr[0] for attr in dataset['attributes']])

print(f"\nDataset shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nFirst 5 rows:")
df.head()

Looking for: /workspaces/PhishTrace/data/raw/Training Dataset.arff
File exists: True

Dataset shape: (11055, 31)

Columns: ['having_IP_Address', 'URL_Length', 'Shortining_Service', 'having_At_Symbol', 'double_slash_redirecting', 'Prefix_Suffix', 'having_Sub_Domain', 'SSLfinal_State', 'Domain_registeration_length', 'Favicon', 'port', 'HTTPS_token', 'Request_URL', 'URL_of_Anchor', 'Links_in_tags', 'SFH', 'Submitting_to_email', 'Abnormal_URL', 'Redirect', 'on_mouseover', 'RightClick', 'popUpWidnow', 'Iframe', 'age_of_domain', 'DNSRecord', 'web_traffic', 'Page_Rank', 'Google_Index', 'Links_pointing_to_page', 'Statistical_report', 'Result']

First 5 rows:


,having_IP_Address,URL_Length,Shortining_Service,having_At_Symbol,double_slash_redirecting,Prefix_Suffix,having_Sub_Domain,SSLfinal_State,Domain_registeration_length,Favicon,...,popUpWidnow,Iframe,age_of_domain,DNSRecord,web_traffic,Page_Rank,Google_Index,Links_pointing_to_page,Statistical_report,Result
0,-1,1,1,1,-1,-1,-1,-1,-1,1,...,1,1,-1,-1,-1,-1,1,1,-1,-1
1,1,1,1,1,1,-1,0,1,-1,1,...,1,1,-1,-1,0,-1,1,1,1,-1
2,1,0,1,1,1,-1,-1,-1,-1,1,...,1,1,1,-1,1,-1,1,0,-1,-1
3,1,0,1,1,1,-1,-1,-1,1,1,...,1,1,-1,-1,1,-1,1,-1,1,-1
4,1,0,-1,1,1,-1,1,1,-1,1,...,-1,1,-1,-1,0,-1,1,1,1,1


## Data Exploration Findings

- **Dataset Source:** UCI Phishing Websites Dataset (ARFF format)
- **Total Samples:** 11,055 URLs
- **Total Features:** 31 (30 predictive features + 1 target label)
- **Data Type:** All features are numeric and pre-encoded (-1, 0, 1)
- **Missing Values:** 0 (Based on the clean output)
- **Target Variable:** The last column (usually named 'Result')
- **Observation:** The dataset is perfectly clean and ready for model training without preprocessing.

In [5]:
# Check the last column (the target)
print("Last column name:", df.columns[-1])
print("\nTarget distribution:")
print(df[df.columns[-1]].value_counts())

Last column name: Result

Target distribution:
Result
1     6157
-1    4898
Name: count, dtype: int64


## Baseline Model Results (Week 1)

### Dataset Split
- Training set: 8,844 samples (80%)
- Test set: 2,211 samples (20%)
- Stratified split to maintain class balance

### Model Comparison

| Model | Accuracy | Precision | Recall | F1-Score |
|-------|----------|-----------|--------|----------|
| Logistic Regression | 92.76% | 93.43% | 90.00% | 91.68% |
| Random Forest | **97.51%** | **97.93%** | **96.43%** | **97.17%** |

### Best Model: Random Forest
- Accuracy: 97.51%
- Correctly classified 2,156 out of 2,211 test URLs
- Only 20 false positives and 35 false negatives

### Confusion Matrix (Random Forest)


### Key Observations
1. Random Forest significantly outperformed Logistic Regression (+4.75% accuracy)
2. Both precision and recall are above 96%, meaning the model is both accurate and reliable
3. Low false positive rate (1.6%) — important for not annoying users
4. Low false negative rate (3.6%) — important for security

### Limitations
- Dataset is pre-engineered, not raw URLs
- May not generalize to real-world URLs without feature extraction
- Performance on live URLs needs validation

In [2]:
# See the exact features the model was trained on
import joblib
import os

# Use path that works from the notebooks/ folder
model_path = '../models/feature_names.pkl'

print(f"Looking for: {os.path.abspath(model_path)}")
print(f"File exists: {os.path.exists(model_path)}")

feature_names = joblib.load(model_path)
print(f"\nModel expects {len(feature_names)} features:")
for i, name in enumerate(feature_names, 1):
    print(f"  {i}. {name}")

Looking for: /workspaces/PhishTrace/models/feature_names.pkl
File exists: True

Model expects 30 features:
  1. having_IP_Address
  2. URL_Length
  3. Shortining_Service
  4. having_At_Symbol
  5. double_slash_redirecting
  6. Prefix_Suffix
  7. having_Sub_Domain
  8. SSLfinal_State
  9. Domain_registeration_length
  10. Favicon
  11. port
  12. HTTPS_token
  13. Request_URL
  14. URL_of_Anchor
  15. Links_in_tags
  16. SFH
  17. Submitting_to_email
  18. Abnormal_URL
  19. Redirect
  20. on_mouseover
  21. RightClick
  22. popUpWidnow
  23. Iframe
  24. age_of_domain
  25. DNSRecord
  26. web_traffic
  27. Page_Rank
  28. Google_Index
  29. Links_pointing_to_page
  30. Statistical_report


## Week 2 Results: Hybrid ML + Heuristic Detection

### The Problem
The ML model alone predicted "SAFE" for obvious phishing URLs because 
some features in our extractor are placeholders (set to 1 = safe).

### The Solution
Added a heuristic override layer:
- IP address detection (+30)
- Suspicious TLD (+25)  
- @ symbol (+20)
- Phishing keywords (+15)
- Excessive hyphens (+15)

If heuristic score ≥ 40, it overrides the ML prediction.

### Results After Fix

| URL | Verdict | Score | ML Prediction |
|-----|---------|-------|---------------|
| https://www.google.com | SAFE | 1/100 | Legitimate |
| https://github.com | SAFE | 2/100 | Legitimate |
| http://192.168.1.1/login/verify | SUSPICIOUS | 45/100 | Legitimate |
| http://secure-login-verify.tk/account/update | PHISHING | 60/100 | Legitimate |

### Key Insight
The ML model alone is insufficient when features are incomplete. 
A hybrid system (ML + heuristics) provides defense-in-depth and 
catches cases the model misses.

### Limitations
- ML model needs more real features (WHOIS, DNS, page content) to improve
- Heuristics can produce false positives for legitimate sites with similar patterns
- Real-world validation still needed